# Experiment 7 — Global–Local ROI Feature Fusion (MiT-B1 + CORN) — Colab Run

Adapts the global–local fusion architecture from Bayraktar et al. (2026, *Integrating Localization, Feature Fusion, and Attention for Pressure Ulcer Staging from Clinical Images*, JIIM, Fig. 4 -- NOT the attention-enhanced Fig. 5 model). `logits = fusion_head(concat(F_global, F_local))`:

- `F_global` = GAP of the **frozen, trained RGB MiT-B1+CORN baseline**'s final-stage feature map (`[B,512,7,7] -> [B,512]`), loaded per fold -- not retrained from ImageNet.
- `F_local` = a projected, `roi_align`-pooled **third-stage** feature map (`[B,320,14,14]`) over the predicted wound ROI box from `results/roi_manifest.csv` (the leakage-free Task 1 ensemble's largest connected wound component, +8px margin, full-image fallback on the 41/1091 detection failures -- generated locally already, committed to git, not regenerated here).
- `fusion_head` is the baseline's own CornHead **widened** from 512->640 inputs: the 512 global columns + bias are copied exactly, the 128 local columns zero-initialized -- at construction `fusion_logits == baseline_logits` EXACTLY (`torch.equal`, verified in `scripts/test_model.py` and again here against the real checkpoint before training).

Phase A: MiT-B1 backbone frozen for the entire run; only `local_proj` + `fusion_head` train (~0.043M trainable params out of 13.2M total).

**Local sanity run already done** (this notebook reproduces it on Colab for the official record): accuracy 0.7983/0.8204/0.8174, macro-F1 0.8008/0.8224/0.8193, QWK 0.9114/0.9249/0.9030 (fold0/1/2) -- essentially matching the RGB baseline (0.8168±0.0150 / 0.8182±0.0138 / 0.9138±0.0164), no significant change. The ROI ablation (Step 8) found **zero prediction flips** between predicted/full-image/shuffled ROI boxes in all 3 folds -- the local branch learned a real but tiny weight (its fusion-head columns are ~20-35x smaller in magnitude than the global columns), too small to ever move a CORN decision. Re-run here to get fast-GPU numbers and re-confirm before treating this as final.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the PIID staging dataset

Same `wound-stage-classification.zip` task_3's own notebook uses -- this experiment reads RGB images directly from it (no tissue-probability dataset needed; `results/roi_manifest.csv` already has the ROI boxes, committed to git, not regenerated here).

In [ ]:
import glob

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-stage-classification"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(candidates[0], DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/task_3/requirements.txt

## 5. RGB baseline checkpoints (fold0/1/2)

First tries to copy them from `drive/MyDrive/thesis/task_3_outputs/checkpoints/` (task_3's own Colab notebook output location). Only whichever folds are still missing after that get trained -- task_3's OWN, unmodified ablation script, restricted to `mit_b1__corn` (not a different/new RGB model for this experiment; fast if needed: ~90s/fold per `task_3/results/task_3_outputs/results/ablation_results.csv`).

In [ ]:
import glob
import os
import shutil

os.chdir(f"{REPO_DIR}/task_3")
CKPT_DIR = "results/task_3_outputs/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DRIVE_CKPT_DIR = "/content/drive/MyDrive/thesis/task_3_outputs"
all_drive_pt_files = glob.glob(f"{DRIVE_CKPT_DIR}/**/*.pt", recursive=True)
print(f"{len(all_drive_pt_files)} .pt file(s) found under {DRIVE_CKPT_DIR}")

for fold in range(3):
    local_path = f"{CKPT_DIR}/mit_b1__corn_fold{fold}.pt"
    if os.path.exists(local_path):
        continue
    candidates = [
        p for p in all_drive_pt_files
        if "mit_b1" in p.lower() and "corn" in p.lower() and "softmax" not in p.lower()
        and f"fold{fold}" in p.lower().replace("_fold", "fold").replace("-fold", "fold")
    ]
    if candidates:
        shutil.copy(candidates[0], local_path)
        print(f"fold{fold}: copied from Drive -> {candidates[0]}")
    else:
        print(f"fold{fold}: no match found under {DRIVE_CKPT_DIR}")

missing = [f for f in range(3) if not os.path.exists(f"{CKPT_DIR}/mit_b1__corn_fold{f}.pt")]
print("still missing after checking Drive:", missing)

if missing:
    !python check_compatibility.py
    !python run_ablation.py --configs mit_b1__corn --batch 16
else:
    print("all 3 RGB baseline checkpoints present, skipping training")

## 6. Architecture sanity check

No checkpoint needed -- `torch.equal` zero-init verification, ROI-content-sensitivity checks, freeze/unfreeze toggles.

In [ ]:
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp7_global_local_roi_fusion/scripts")
!python test_model.py

## 7. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp7_global_local_roi_fusion_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 8. Train Phase A — all 3 folds

Each fold loads its OWN matching RGB baseline checkpoint, verifies `fusion_logits == baseline_logits` exactly before training (stops immediately if not), then trains only `local_proj` + `fusion_head` with real early stopping (`patience` from `configs/base.yaml`). MiT-B1 backbone frozen the whole run (verified via an unchanged-`f_global` probe-batch check after training).

In [ ]:
!python train.py --workers 4

## 9. Check results

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())
print()
print("RGB baseline for reference: accuracy=0.8168+-0.0150 macro_f1=0.8182+-0.0138 qwk=0.9138+-0.0164")

## 10. ROI contribution ablation (Step 8)

Inference-only, same trained weights: A (predicted ROI) vs B (full-image ROI) vs C (shuffled ROI from another image). Local run found **zero** prediction flips in all 3 folds -- re-confirm here. Reads checkpoints from `../checkpoints/` (the config still points at Drive from cell 7, so this reads the just-trained Colab checkpoints, not the local sanity-run ones).

In [ ]:
!python roi_ablation.py

## 11. Evaluation and error analysis (Step 9)

Per-class precision/recall/F1, confusion matrices, failed-localization subset, and the paired RGB-vs-fusion comparison.

In [ ]:
!python error_analysis.py